In [23]:
import warnings
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.ets import AutoETS

from sktime.forecasting.model_selection import ForecastingGridSearchCV
from sktime.split import ExpandingWindowSplitter

warnings.filterwarnings("ignore")

In [26]:
def evaluate_dataset(csv_path: str):
  print("=" * 100)
  print(csv_path)
  print("=" * 100)

  df = pd.read_csv(csv_path)
  df["Date"] = pd.to_datetime(df["Date"])
  df = df.sort_values("Date")
  y = df.set_index("Date")["Revenue"].astype(float)

  y_train = y.loc[:"2010-10-31"]
  y_test = y.loc["2010-11-01":]

  fh = ForecastingHorizon(
    np.arange(1, len(y_test) + 1),
    is_relative=True
  )

  cv = ExpandingWindowSplitter(
    initial_window=int(len(y_train) * 0.60),
    step_length=30,
    fh=np.arange(1, 31)
  )

  print("\nTRAIN SIZE:", len(y_train))
  print("\nTEST SIZE :", len(y_test))

  # =========
  # AUTOARIMA
  # =========
  print("\nSearching AutoARIMA...")

  arima = AutoARIMA(
    seasonal=True,
    suppress_warnings=True,
    error_action="ignore"
  )

  arima_grid = {
    "sp": [7, 30],
    "information_criterion": ["aic", "bic"],
    "max_p": [3, 5],
    "max_q": [3, 5]
  }

  arima_search = ForecastingGridSearchCV(
    forecaster=arima,
    param_grid=arima_grid,
    cv=cv,
    strategy="refit"
  )

  arima_search.fit(y_train)
  best_arima = arima_search.best_forecaster_

  print("\nBEST AUTOARIMA PARAMS")
  print(arima_search.best_params_)

  best_arima.fit(y_train)
  arima_pred = best_arima.predict(fh)

  arima_mae = mean_absolute_error(y_test, arima_pred)
  arima_rmse = np.sqrt(mean_squared_error(y_test, arima_pred))

  # =======
  # AUTOETS
  # =======
  print("Searching AutoETS...")

  ets = AutoETS(auto=True)

  ets_grid = {
    "sp": [7, 30],
    "additive_only": [True, False],
    "allow_multiplicative_trend": [True, False]
  }

  ets_search = ForecastingGridSearchCV(
    forecaster=ets,
    param_grid=ets_grid,
    cv=cv,
    strategy="refit"
  )

  ets_search.fit(y_train)
  best_ets = ets_search.best_forecaster_

  print("\nBEST AUTOETS PARAMS")
  print(ets_search.best_params_)

  best_ets.fit(y_train)
  ets_pred = best_ets.predict(fh)
  ets_mae = mean_absolute_error(y_test, ets_pred)
  ets_rmse = np.sqrt(mean_squared_error(y_test, ets_pred))

  # =======
  # RESULTS
  # =======
  results = pd.DataFrame({
    "Actual": y_test,
    "AutoARIMA": arima_pred,
    "AutoETS": ets_pred
  })

  print("\nFORECASTS")
  print(results)

  print("\nAUTOARIMA")
  print(f"MAE  = {arima_mae:.4f}")
  print(f"RMSE = {arima_rmse:.4f}")

  print("\nAUTOETS")
  print(f"MAE  = {ets_mae:.4f}")
  print(f"RMSE = {ets_rmse:.4f}")

  results.to_csv(csv_path.replace(".csv", "_forecasts.csv"), index=True)

  return {
    "dataset": csv_path,
    "arima_mae": arima_mae,
    "arima_rmse": arima_rmse,
    "ets_mae": ets_mae,
    "ets_rmse": ets_rmse,
    "arima_params": arima_search.best_params_,
    "ets_params": ets_search.best_params_,
  }

In [ ]:
summary = []
for dataset in ["product.csv", "country.csv", "customer.csv"]:
  summary.append(evaluate_dataset("data/" + dataset))

summary = pd.DataFrame(summary)

print()
print("=" * 100)
print("FINAL SUMMARY")
print("=" * 100)
print(summary)

data/product.csv

TRAIN SIZE: 165

TEST SIZE : 29

Searching AutoARIMA...


/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return x.to_period(freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x.to_period(freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return x.to_period(freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay


BEST AUTOARIMA PARAMS
{'information_criterion': 'bic', 'max_p': 3, 'max_q': 3, 'sp': 7}


/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return x.to_period(freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x.to_period(freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/adapters/_statsmodels.py:69: UserWarning: Warning: time series is not strictly positive, multiplicative components are omitted
  self._fit_forecaster(y, X)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home

Searching AutoETS...


/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/t


BEST AUTOETS PARAMS
{'additive_only': True, 'allow_multiplicative_trend': True, 'sp': 7}


/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency B will be used.
  self._init_dates(dates, freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/statsmodels/tsa/base/t


FORECASTS
            Actual   AutoARIMA     AutoETS
2010-11-01   25.50  217.953007  248.901114
2010-11-02  114.75  214.468573  248.901114
2010-11-03  216.75  211.614756  248.901114
2010-11-04  446.25  209.277428  248.901114
2010-11-05  267.75  207.363112  248.901114
2010-11-08  102.00  205.795251  248.901114
2010-11-09  193.20  204.511145  248.901114
2010-11-10  765.00  203.459438  248.901114
2010-11-11  280.50  202.598070  248.901114
2010-11-12   89.25  201.892593  248.901114
2010-11-15  306.00  201.314795  248.901114
2010-11-16  242.25  200.841568  248.901114
2010-11-17   63.75  200.453986  248.901114
2010-11-18  191.25  200.136549  248.901114
2010-11-19  114.75  199.876562  248.901114
2010-11-22  204.00  199.663628  248.901114
2010-11-23  293.25  199.489231  248.901114
2010-11-24  306.00  199.346397  248.901114
2010-11-25  395.25  199.229413  248.901114
2010-11-26   76.50  199.133601  248.901114
2010-11-29  191.25  199.055129  248.901114
2010-11-30  331.50  198.990859  248.901114


/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return x.to_period(freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay freq instead.
  return x.to_period(freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: PeriodDtype[B] is deprecated and will be removed in a future version. Use a DatetimeIndex with freq='B' instead
  return x.to_period(freq)
/home/wesley/Work/engine/eniac/.venv/lib/python3.12/site-packages/sktime/forecasting/base/_fh.py:946: FutureWarning: Period with BDay freq is deprecated and will be removed in a future version. Use a DatetimeIndex with BDay